# MNIST Handwritten Digit Classification
## From a Simple Artificial Neural Network to a Convolutional Neural Network

This notebook demonstrates how neural networks can classify handwritten digits from **0 to 9**.

### Learning objectives

By the end of this notebook, you should be able to:

1. Load and inspect the MNIST dataset.
2. Prepare image data for a neural network.
3. Build a **simple Artificial Neural Network (ANN)**.
4. Evaluate the ANN using unseen test data.
5. Build a **Convolutional Neural Network (CNN)** to improve image-classification accuracy.
6. Upload your own handwritten digits.
7. Convert uploaded images into the same array format used by MNIST.
8. Use the trained model to classify your handwritten numbers.

> **Why two models?**  
> The first model is intentionally simple so we can connect it to the basic ANN concepts discussed in the lecture.  
> The second model uses convolutional layers, which are better suited for image data because they can learn local visual patterns such as edges, curves, and shapes.

## 1. Import the required libraries

Google Colab already includes most of the libraries used in this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

## 2. Load the MNIST dataset

The **MNIST** dataset contains grayscale images of handwritten digits from **0 to 9**.

Each image has:

- Width: **28 pixels**
- Height: **28 pixels**
- Color channels: **1 grayscale channel**

The dataset is already separated into training and testing sets.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

## 3. Inspect sample images

Each image is represented as a **28 × 28 matrix** of pixel values.

Pixel values range from:

- `0` = black
- `255` = white

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Inspect the image as an array

A neural network does not directly "see" an image.  
It receives numbers.

Below is the array representation of one MNIST image.

In [ ]:
print("Label:", y_train[0])
print("Image shape:", x_train[0].shape)
print(x_train[0])

## 5. Normalize pixel values

Neural networks generally train more effectively when input values are kept within a smaller range.

We convert the original pixel values from:

`0 to 255`

into:

`0.0 to 1.0`

In [ ]:
x_train_norm = x_train.astype("float32") / 255.0
x_test_norm = x_test.astype("float32") / 255.0

print("Minimum:", x_train_norm.min())
print("Maximum:", x_train_norm.max())

# Part A — Simple Artificial Neural Network

A regular dense ANN expects a one-dimensional vector rather than a 2D image.

We will therefore convert each **28 × 28 image** into a vector containing:

`28 × 28 = 784 input values`

### Architecture

**Input image (28 × 28)**  
↓  
**Flatten → 784 values**  
↓  
**Dense hidden layer: 128 neurons, ReLU**  
↓  
**Output layer: 10 neurons, Softmax**

The 10 output neurons represent digits **0 through 9**.

## 6. Build the simple ANN

In [ ]:
ann_model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

ann_model.summary()

### Why Softmax?

The output layer has 10 possible classes.

Softmax converts the output values into a probability distribution where all probabilities add up to approximately **1.0**.

## 7. Compile the ANN

We use:

- **Adam** optimizer
- **Sparse categorical cross-entropy** loss
- **Accuracy** as the evaluation metric

`SparseCategoricalCrossentropy` is appropriate because the labels are stored as integers such as `0`, `1`, `2`, ..., `9`.

In [ ]:
ann_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 8. Train the ANN

In [ ]:
ann_history = ann_model.fit(
    x_train_norm,
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)

## 9. Visualize the ANN training history

In [ ]:
history_df = pd.DataFrame(ann_history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df.index + 1, history_df["accuracy"], marker="o", label="Training accuracy")
plt.plot(history_df.index + 1, history_df["val_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Simple ANN: Training vs Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Evaluate the ANN on unseen test data

In [ ]:
ann_test_loss, ann_test_accuracy = ann_model.evaluate(x_test_norm, y_test, verbose=0)

print(f"ANN test loss: {ann_test_loss:.4f}")
print(f"ANN test accuracy: {ann_test_accuracy:.4%}")

## 11. Inspect some ANN predictions

In [ ]:
ann_probabilities = ann_model.predict(x_test_norm[:10], verbose=0)
ann_predictions = np.argmax(ann_probabilities, axis=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i], cmap="gray")
    ax.set_title(
        f"Actual: {y_test[i]}\nPredicted: {ann_predictions[i]}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

# Part B — Improve Accuracy with a Convolutional Neural Network

The simple ANN treats the image as 784 independent numbers after flattening.

However, images contain **spatial structure**:

- neighboring pixels are related;
- strokes form edges;
- edges form curves and shapes;
- shapes help distinguish digits.

A **Convolutional Neural Network (CNN)** preserves this spatial information and learns useful image features automatically.

### Improved model architecture

**28 × 28 × 1 image**  
↓  
**Conv2D: 32 filters**  
↓  
**MaxPooling**  
↓  
**Conv2D: 64 filters**  
↓  
**MaxPooling**  
↓  
**Flatten**  
↓  
**Dense: 128 neurons**  
↓  
**Dropout**  
↓  
**10-class Softmax output**

## Before We Build the CNN: What Do Convolution and Max Pooling Do?

Before writing the CNN, let us trace **one MNIST image** through the first two operations:

```python
layers.Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same")
layers.MaxPooling2D(pool_size=(2, 2))
```

The overall transformation is:

```text
Input: 28 × 28 × 1
        ↓
Conv2D: 32 different 3 × 3 filters
        ↓
28 × 28 × 32
        ↓
MaxPooling2D: 2 × 2
        ↓
14 × 14 × 32
```

The final `1` in `(28, 28, 1)` is the grayscale channel. Unlike the simple ANN, **we do not flatten the image immediately** because we want the CNN to preserve relationships between neighboring pixels.

### Step 1 — Start with a 28 × 28 image

A real MNIST image contains **784 pixel values**. A simplified portion might look like:

```text
[
 [0,0,0,0,0,0,0,0, ... 28 values ...],
 [0,0,0,0,0,0,0,0, ...],
 [0,0,0,1,1,1,1,0, ...],
 [0,0,0,0,0,0,1,0, ...],
 [0,0,0,0,0,1,0,0, ...],
 ...
 28 rows total
]
```

Its shape is:

```text
28 rows × 28 columns × 1 grayscale channel
= (28, 28, 1)
```

### Step 2 — A 3 × 3 convolution filter examines a small region

Suppose **one** filter contains these illustrative weights:

```text
[ 1,  0, -1]
[ 1,  0, -1]
[ 1,  0, -1]
```

In the real model these weights are **learned during training**.

Suppose that filter is currently over this 3 × 3 image region:

```text
Image region          Filter

[1, 0, 0]             [ 1,  0, -1]
[1, 0, 0]      ×      [ 1,  0, -1]
[1, 0, 0]             [ 1,  0, -1]
```

Element-wise multiplication and addition gives:

```text
(1×1) + (0×0) + (0×-1)
+ (1×1) + (0×0) + (0×-1)
+ (1×1) + (0×0) + (0×-1)

= 3
```

Ignoring bias for this illustration:

```text
z = 3
ReLU(3) = 3
```

So the corresponding position in the resulting **feature map** gets the value `3`.

The same filter then slides across other locations in the image and repeats the calculation.

### Step 3 — 32 filters create 32 feature maps

One filter scanning the image produces one feature map:

```text
28 × 28 image
      ↓
3 × 3 Filter #1
      ↓
28 × 28 Feature Map #1
```

But our layer says:

```python
Conv2D(32, kernel_size=(3, 3), padding="same")
```

So the CNN learns **32 different filters**:

```text
Filter #1  → 28 × 28 feature map
Filter #2  → 28 × 28 feature map
Filter #3  → 28 × 28 feature map
...
Filter #32 → 28 × 28 feature map
```

Stacking the feature maps gives:

```text
28 × 28 × 32
```

`padding="same"` keeps the spatial dimensions at `28 × 28`.

So convolution is not primarily compressing the image. It is creating **multiple learned representations of visual features**.

### Step 4 — MaxPooling2D reduces each feature map

Suppose a small `4 × 4` portion of one feature map is:

```text
0.1  0.8 | 0.2  0.4
0.3  0.6 | 0.9  0.1
---------+---------
0.7  0.2 | 0.4  0.3
0.1  0.5 | 0.8  0.6
```

With:

```python
MaxPooling2D(pool_size=(2, 2))
```

we keep the maximum value from each `2 × 2` region:

```text
max(0.1, 0.8, 0.3, 0.6) = 0.8
max(0.2, 0.4, 0.9, 0.1) = 0.9
max(0.7, 0.2, 0.1, 0.5) = 0.7
max(0.4, 0.3, 0.8, 0.6) = 0.8
```

Therefore:

```text
Before pooling             After pooling

0.1 0.8 0.2 0.4
0.3 0.6 0.9 0.1              0.8 0.9
                    →         0.7 0.8
0.7 0.2 0.4 0.3
0.1 0.5 0.8 0.6

     4 × 4                       2 × 2
```

Across all 32 feature maps:

```text
28 × 28 × 32
      ↓
MaxPooling2D(2 × 2)
      ↓
14 × 14 × 32
```

Pooling reduces **height and width**, but it does not change the number of feature maps.

### Complete first-stage transformation

```text
ONE MNIST IMAGE
──────────────────────────────
28 × 28 × 1
784 original pixel values

            ↓

CONV2D
──────────────────────────────
32 learned filters
Each filter = 3 × 3

Filter #1  → 28 × 28
Filter #2  → 28 × 28
...
Filter #32 → 28 × 28

Combined:
28 × 28 × 32
25,088 feature values

            ↓

MAXPOOLING2D
──────────────────────────────
Each 2 × 2 region becomes
its maximum value

28 × 28 × 32
      ↓
14 × 14 × 32
6,272 feature values
```

A useful way to remember the two operations:

> **Convolution:** “What visual features can I find?”

> **Max pooling:** “Can I keep the strongest evidence while reducing the spatial size?”

The convolution filters contain **trainable weights**, just like the neurons in the earlier ANN. The key difference is that the same small filter is applied repeatedly across different locations in the image.

### Verify the shapes in Python

The following example passes a dummy image through one convolution and one pooling layer. The filters are not trained yet; this is simply to demonstrate how the tensor shape changes.

In [ ]:
sample_image = np.zeros((1, 28, 28, 1), dtype=np.float32)

sample_conv = layers.Conv2D(
    32,
    kernel_size=(3, 3),
    activation="relu",
    padding="same"
)

sample_pool = layers.MaxPooling2D(pool_size=(2, 2))

conv_output = sample_conv(sample_image)
pool_output = sample_pool(conv_output)

print("Input shape:        ", sample_image.shape)
print("After Conv2D:       ", conv_output.shape)
print("After MaxPooling2D: ", pool_output.shape)

## 12. Reshape the data for the CNN

A CNN expects the image dimensions plus the number of color channels.

MNIST is grayscale, so each image has **1 channel**.

In [ ]:
x_train_cnn = np.expand_dims(x_train_norm, axis=-1)
x_test_cnn = np.expand_dims(x_test_norm, axis=-1)

print("CNN training shape:", x_train_cnn.shape)
print("CNN test shape:", x_test_cnn.shape)

## 13. Build the CNN

In [ ]:
cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(10, activation="softmax")
])

cnn_model.summary()

## 14. Compile the CNN

In [ ]:
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 15. Train the CNN

An **EarlyStopping** callback is included so training can stop when validation accuracy no longer improves.

This helps reduce unnecessary training and can reduce overfitting.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True
)

cnn_history = cnn_model.fit(
    x_train_cnn,
    y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

## 16. Visualize CNN training history

In [ ]:
cnn_history_df = pd.DataFrame(cnn_history.history)

plt.figure(figsize=(8, 5))
plt.plot(cnn_history_df.index + 1, cnn_history_df["accuracy"], marker="o", label="Training accuracy")
plt.plot(cnn_history_df.index + 1, cnn_history_df["val_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN: Training vs Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 17. Evaluate the CNN

In [ ]:
cnn_test_loss, cnn_test_accuracy = cnn_model.evaluate(x_test_cnn, y_test, verbose=0)

print(f"CNN test loss: {cnn_test_loss:.4f}")
print(f"CNN test accuracy: {cnn_test_accuracy:.4%}")

## 18. Compare the two models

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Simple ANN", "CNN"],
    "Test Accuracy": [ann_test_accuracy, cnn_test_accuracy],
    "Test Loss": [ann_test_loss, cnn_test_loss]
})

comparison

## 19. Create a confusion matrix for the CNN

A confusion matrix helps us inspect which digits are commonly confused with each other.

In [ ]:
cnn_test_probabilities = cnn_model.predict(x_test_cnn, verbose=0)
cnn_test_predictions = np.argmax(cnn_test_probabilities, axis=1)

cm = confusion_matrix(y_test, cnn_test_predictions)

fig, ax = plt.subplots(figsize=(9, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
plt.title("CNN Confusion Matrix")
plt.show()

# Part C — Test the Model Using Your Own Handwriting

The model was trained using MNIST images, so uploaded images must be converted into a similar format:

1. Grayscale
2. High contrast
3. White digit on black background
4. Cropped around the digit
5. Resized while preserving aspect ratio
6. Centered inside a **28 × 28** image
7. Pixel values normalized to `0.0–1.0`

The helper functions below perform these steps automatically.

You can test the model in either of these ways:

### Option 1 — Upload one image per digit

For example:

- `0.jpg`
- `1.jpg`
- ...
- `9.jpg`

### Option 2 — Upload one photo containing several handwritten digits

For example, write:

`0 1 2 3 4 5 6 7 8 9`

on white paper, take a clear photo, and upload it.

The notebook will attempt to detect and separate the individual digits automatically.

## 20. Import image-processing tools

In [ ]:
import cv2
from google.colab import files
from PIL import Image
import io
import math

## 21. Helper function: convert one digit into MNIST format

This function:

- receives a grayscale digit image;
- finds the digit;
- crops unnecessary background;
- resizes the digit;
- places it in the center of a 28 × 28 image;
- normalizes the pixels.

The result is compatible with the CNN.

In [ ]:
def prepare_digit_for_mnist(gray_image, show_steps=False):
    """
    Convert a grayscale image containing one handwritten digit
    into MNIST-like 28x28 format.

    Returns:
        normalized_image: shape (28, 28), values 0.0-1.0
    """

    # Ensure uint8
    gray = gray_image.astype(np.uint8)

    # Slight blur can reduce camera/image noise
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # Convert to binary using Otsu thresholding
    _, binary = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # MNIST uses light digits on dark background.
    # If the image background is mostly white, invert it.
    if np.mean(binary) > 127:
        binary = cv2.bitwise_not(binary)

    # Find non-zero digit pixels
    coords = cv2.findNonZero(binary)

    if coords is None:
        raise ValueError("No digit could be detected in the image.")

    x, y, w, h = cv2.boundingRect(coords)
    digit = binary[y:y+h, x:x+w]

    # Resize longest dimension to 20 pixels, similar to MNIST centering
    target_size = 20

    if h > w:
        new_h = target_size
        new_w = max(1, int(round(w * target_size / h)))
    else:
        new_w = target_size
        new_h = max(1, int(round(h * target_size / w)))

    resized = cv2.resize(
        digit,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    # Place resized digit in the center of a 28x28 canvas
    canvas = np.zeros((28, 28), dtype=np.uint8)

    x_offset = (28 - new_w) // 2
    y_offset = (28 - new_h) // 2

    canvas[
        y_offset:y_offset + new_h,
        x_offset:x_offset + new_w
    ] = resized

    normalized_image = canvas.astype("float32") / 255.0

    if show_steps:
        fig, axes = plt.subplots(1, 3, figsize=(9, 3))
        axes[0].imshow(gray_image, cmap="gray")
        axes[0].set_title("Original")
        axes[1].imshow(binary, cmap="gray")
        axes[1].set_title("Thresholded")
        axes[2].imshow(normalized_image, cmap="gray")
        axes[2].set_title("MNIST-ready")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    return normalized_image

## 22. Helper function: classify a single processed digit

In [ ]:
def predict_digit(processed_digit, model=cnn_model):
    model_input = processed_digit.reshape(1, 28, 28, 1)

    probabilities = model.predict(model_input, verbose=0)[0]
    predicted_digit = int(np.argmax(probabilities))
    confidence = float(np.max(probabilities))

    return predicted_digit, confidence, probabilities

## 23. Upload and classify individual digit images

Use this section if you have **one digit per image**.

You can upload multiple image files at the same time.

In [ ]:
uploaded = files.upload()

individual_results = []

for filename, file_bytes in uploaded.items():
    image = Image.open(io.BytesIO(file_bytes)).convert("L")
    gray = np.array(image)

    processed = prepare_digit_for_mnist(gray)
    predicted_digit, confidence, probabilities = predict_digit(processed)

    individual_results.append({
        "filename": filename,
        "processed": processed,
        "prediction": predicted_digit,
        "confidence": confidence,
        "probabilities": probabilities
    })

if individual_results:
    cols = min(5, len(individual_results))
    rows = math.ceil(len(individual_results) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, result in zip(axes, individual_results):
        ax.imshow(result["processed"], cmap="gray")
        ax.set_title(
            f'{result["filename"]}\n'
            f'Prediction: {result["prediction"]}\n'
            f'Confidence: {result["confidence"]:.1%}'
        )
        ax.axis("off")

    for ax in axes[len(individual_results):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# Optional: Classify Several Digits from One Photo

This section attempts to detect multiple handwritten digits from a single image.

For best results:

- use white paper;
- use a dark pen or marker;
- write digits separately;
- leave visible space between digits;
- avoid letting digits touch;
- take the photo as straight as possible;
- use good lighting.

Example:

`0   1   2   3   4   5   6   7   8   9`

## 24. Function to detect multiple digits in one photo

In [ ]:
def extract_digits_from_photo(image_rgb, min_area=100):
    """
    Detect separate handwritten digits from a photo.

    Returns:
        processed_digits: list of 28x28 normalized MNIST-like arrays
        boxes: bounding boxes (x, y, w, h)
        thresholded: thresholded image used for contour detection
    """

    if image_rgb.ndim == 2:
        gray = image_rgb
    else:
        gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    gray_blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    _, thresh = cv2.threshold(
        gray_blurred, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # We want digits to be white and the background black
    if np.mean(thresh) > 127:
        thresh = cv2.bitwise_not(thresh)

    # A very small dilation helps connect broken pen strokes
    kernel = np.ones((2, 2), np.uint8)
    contour_image = cv2.dilate(thresh, kernel, iterations=1)

    contours, _ = cv2.findContours(
        contour_image,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h

        # Ignore tiny noise
        if area >= min_area and h >= 10 and w >= 3:
            boxes.append((x, y, w, h))

    # Sort primarily from left to right
    boxes = sorted(boxes, key=lambda b: b[0])

    processed_digits = []

    for x, y, w, h in boxes:
        # Add a small margin around each detected digit
        margin = 8

        x1 = max(0, x - margin)
        y1 = max(0, y - margin)
        x2 = min(gray.shape[1], x + w + margin)
        y2 = min(gray.shape[0], y + h + margin)

        crop = gray[y1:y2, x1:x2]

        try:
            processed = prepare_digit_for_mnist(crop)
            processed_digits.append(processed)
        except ValueError:
            pass

    return processed_digits, boxes, thresh

## 25. Upload one photo containing several digits

In [ ]:
uploaded_sheet = files.upload()

sheet_filename = next(iter(uploaded_sheet))
sheet_bytes = uploaded_sheet[sheet_filename]

sheet_image = Image.open(io.BytesIO(sheet_bytes)).convert("RGB")
sheet_rgb = np.array(sheet_image)

digits, boxes, thresholded = extract_digits_from_photo(sheet_rgb)

print(f"Detected {len(digits)} digit region(s).")

plt.figure(figsize=(14, 6))
plt.imshow(sheet_rgb)
plt.title("Uploaded handwritten digits")
plt.axis("off")
plt.show()

plt.figure(figsize=(14, 6))
plt.imshow(thresholded, cmap="gray")
plt.title("Thresholded image used for digit detection")
plt.axis("off")
plt.show()

## 26. Classify the detected digits

In [ ]:
sheet_results = []

for i, digit in enumerate(digits):
    predicted_digit, confidence, probabilities = predict_digit(digit)

    sheet_results.append({
        "position": i + 1,
        "processed": digit,
        "prediction": predicted_digit,
        "confidence": confidence
    })

if sheet_results:
    cols = min(10, len(sheet_results))
    rows = math.ceil(len(sheet_results) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 3 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, result in zip(axes, sheet_results):
        ax.imshow(result["processed"], cmap="gray")
        ax.set_title(
            f'#{result["position"]}\n'
            f'Pred: {result["prediction"]}\n'
            f'{result["confidence"]:.1%}'
        )
        ax.axis("off")

    for ax in axes[len(sheet_results):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    predicted_sequence = "".join(
        str(result["prediction"]) for result in sheet_results
    )

    print("Predicted sequence:", predicted_sequence)
else:
    print("No usable digits were detected.")

## 27. Inspect prediction probabilities for one handwritten digit

This helps us see not only the predicted class, but also how strongly the model considered the other classes.

In [ ]:
# Change this index if you want to inspect another digit.
digit_index = 0

if sheet_results:
    digit = sheet_results[digit_index]["processed"]
    predicted_digit, confidence, probabilities = predict_digit(digit)

    plt.figure(figsize=(8, 4))
    plt.bar(range(10), probabilities)
    plt.xticks(range(10))
    plt.xlabel("Digit")
    plt.ylabel("Predicted probability")
    plt.title(
        f"Prediction probabilities — predicted digit: {predicted_digit}"
    )
    plt.show()
else:
    print("Run the multi-digit upload section first.")

# Discussion Questions

1. Why does the ANN need to flatten the 28 × 28 image into 784 values?
2. What information might be lost when the image is flattened?
3. Why is a CNN usually more suitable for image classification?
4. What does a convolutional filter learn from an image?
5. What is the purpose of max pooling?
6. Why does the final layer contain exactly 10 neurons?
7. Why is Softmax used in the output layer?
8. What does the model's confidence score represent?
9. Why might the model perform worse on your own handwriting than on the MNIST test dataset?
10. What preprocessing changes could improve predictions on real handwritten photographs?

# Key Takeaways

- Images are represented as numerical arrays.
- A simple ANN can classify MNIST digits reasonably well after flattening the image.
- Flattening removes the original 2D spatial relationship between pixels.
- CNNs are designed to learn spatial image features and generally perform better on image-classification tasks.
- Real-world images require preprocessing so they resemble the data used during model training.
- Even a high-accuracy test model can make mistakes when the real input looks different from the original training data.

### Important practical lesson

Model performance depends not only on the neural-network architecture, but also on whether **production input data resembles the training data**.

Your handwritten phone photo is a simple example of **data distribution shift**:

- MNIST images are clean and centered.
- Phone photos may contain shadows, perspective distortion, different stroke thicknesses, or off-center digits.

This is why preprocessing and representative training data are important when deploying AI solutions.